# Task 4: Fine-tuning the bi-encoder

Until now every model was used as downloaded. Here the general embedder is trained on our own training claims, twice:

- **random negatives**: the wrong abstracts shown during training are drawn at random from the corpus
- **hard negatives**: the wrong abstracts are ones that BM25 / the dense models from tasks 1-2 ranked highly for that claim, but that are not gold

Everything else (base model, loss, learning rate, epochs, seed, batch order) is identical, so any difference between the two runs comes from the negatives.

Needs the zip from notebook 3. GPU runtime (T4), roughly 20-25 min with bge-large.

In [1]:
# upload NLP_Inter_IIT.zip (the one notebook 3 downloaded in its last cell) and unpack it
# on colab this opens the file picker. If the folder is already here (rerun, or running locally) nothing happens.
import zipfile
from pathlib import Path

def find_root():
    # folder that contains data/, wherever the notebook was opened from
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd() / "NLP_Inter_IIT"]:
        if (p / "data" / "corpus.jsonl").exists():
            return p

if find_root() is None:
    zips = list(Path.cwd().glob("*.zip"))        # zip already dragged into the colab file panel
    if not zips:
        from google.colab import files
        zips = [Path(name) for name in files.upload()]
    newest = max(zips, key=lambda z: z.stat().st_mtime)
    zipfile.ZipFile(newest).extractall(".")
    print("unpacked", newest.name)

ROOT = find_root()
assert ROOT is not None, "data/corpus.jsonl not found, upload NLP_Inter_IIT.zip"
assert (ROOT / "work" / "scores_hybrid_val.npy").exists(), "this zip has no notebook 3 outputs: upload the zip that notebook 3 downloaded in its last cell"
print("project folder:", ROOT)
print(sorted(p.name for p in (ROOT / "predictions").iterdir()))

Saving NLP_Inter_IIT_4.zip to NLP_Inter_IIT_4.zip
unpacked NLP_Inter_IIT_4.zip
project folder: /content/NLP_Inter_IIT
['t1_bm25.trec', 't2_general.trec', 't2_scientific.trec', 't3_hybrid.trec', 't3_rerank.trec', 'val_bm25.trec', 'val_general.trec']


In [2]:
%pip install -q transformers

In [3]:
import json, math, random, sys, time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

DATA, PRED, WORK = ROOT / "data", ROOT / "predictions", ROOT / "work"

SEED = 42
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "none (cpu)")

cuda | Tesla T4


## Data and helpers (same as before)

In [4]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

corpus = sorted(read_jsonl(DATA / "corpus.jsonl"), key=lambda d: d["doc_id"])
doc_ids = np.array([d["doc_id"] for d in corpus])
doc_index = {d["doc_id"]: i for i, d in enumerate(corpus)}           # doc_id -> column in the score matrices
doc_texts = [d["title"] + " " + " ".join(d["abstract"]) for d in corpus]
claims = {s: sorted(read_jsonl(DATA / f"{s}_claims.jsonl"), key=lambda c: c["id"]) for s in ["train", "val", "eval"]}

TOP_K = 100

def gold_docs(split):
    return {c["id"]: {e["doc_id"] for e in c["evidence"]} for c in claims[split]}

def make_run(scores, split, k=TOP_K):
    order = np.argsort(-scores, axis=1, kind="stable")[:, :k]
    run = {}
    for i, c in enumerate(claims[split]):
        run[c["id"]] = [(int(doc_ids[j]), float(scores[i, j])) for j in order[i]]
    return run

def claim_metrics(run, gold):
    rows = {}
    for cid, g in gold.items():
        if len(g) == 0:
            continue
        ranked = [d for d, _ in run[cid]]
        dcg = sum(1 / math.log2(r + 2) for r, d in enumerate(ranked[:10]) if d in g)
        idcg = sum(1 / math.log2(r + 2) for r in range(min(len(g), 10)))
        rr = 0.0
        for r, d in enumerate(ranked):
            if d in g:
                rr = 1 / (r + 1)
                break
        rows[cid] = {"ndcg10": dcg / idcg, "recall100": len(g & set(ranked)) / len(g), "mrr": rr}
    return pd.DataFrame.from_dict(rows, orient="index")

def evaluate(run, gold):
    return claim_metrics(run, gold).mean().round(4).to_dict()

def write_trec(path, run, name):
    with open(path, "w") as f:
        for cid in sorted(run):
            for rank, (did, score) in enumerate(run[cid], start=1):
                f.write(f"{cid} Q0 {did} {rank} {score:.6f} {name}\n")

results_log = json.load(open(WORK / "results_log.json"))

## Settings

The base model is the general embedder that won in notebook 2 (bge-large). It has 335M parameters, which is a lot for a T4, so gradient checkpointing is on: activations are recomputed during the backward pass instead of being stored, which costs some speed but keeps memory well under the limit.

If you still get an out-of-memory error: set `BATCH_SIZE = 8`, or switch `BASE_MODEL` to `"BAAI/bge-base-en-v1.5"` (then `LR = 2e-5` and `GRAD_CHECKPOINT = False` are fine).

In [5]:
BASE_MODEL = "BAAI/bge-large-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "    # bge convention, same as in notebook 2

EPOCHS = 4
BATCH_SIZE = 16          # claims per step
N_NEG = 2                # explicit negatives per claim (on top of the other abstracts in the batch)
LR = 1e-5
TEMPERATURE = 0.02       # cosine similarities are divided by this before the softmax (value bge was trained with)
WARMUP = 0.1             # share of steps with linear warm-up
TRAIN_DOC_LEN = 256      # abstracts are cut to this many tokens during training (512 when indexing)
QUERY_MAX_LEN, DOC_MAX_LEN = 64, 512
HARD_POOL = 30           # hard negatives are sampled from the top 30 non-gold abstracts
GRAD_CHECKPOINT = True

## Training pairs and negatives

One training example per (claim, gold abstract) pair. Claims without evidence can't be used, since they have no positive.

**Hard negative pool**: for every training claim, the BM25, general and scientific rankings from tasks 1-2 are fused with RRF, gold abstracts are removed, and the top 30 of what is left is the pool. These are abstracts that our own retrievers consider very relevant but that are not labelled as evidence.

**Random pool**: any abstract in the corpus that is not gold for the claim.

In both runs `N_NEG` negatives per claim are drawn from the pool, fresh every epoch.

In [6]:
train_gold = gold_docs("train")
pairs = [(c["id"], c["claim"], e["doc_id"]) for c in claims["train"] for e in c["evidence"]]
print(len(pairs), "training pairs from", len({p[0] for p in pairs}), "claims")

# RRF over the three task 1-2 systems on the training claims
def to_ranks(mat):
    order = np.argsort(-mat, axis=1, kind="stable")
    ranks = np.empty_like(order)
    ranks[np.arange(mat.shape[0])[:, None], order] = np.arange(1, mat.shape[1] + 1)
    return ranks

fused = sum(1 / (60 + to_ranks(np.load(WORK / f"scores_{name}_train.npy"))) for name in ["bm25", "general", "scientific"])
fused_order = np.argsort(-fused, axis=1, kind="stable")[:, :HARD_POOL + 10]

hard_pool = {}
for i, c in enumerate(claims["train"]):
    cands = [int(doc_ids[j]) for j in fused_order[i] if int(doc_ids[j]) not in train_gold[c["id"]]]
    hard_pool[c["id"]] = cands[:HARD_POOL]

all_doc_ids = doc_ids.tolist()

def sample_negatives(mode, cid, rng):
    if mode == "hard":
        return rng.sample(hard_pool[cid], N_NEG)
    out = []
    while len(out) < N_NEG:
        d = rng.choice(all_doc_ids)
        if d not in train_gold[cid] and d not in out:
            out.append(d)
    return out

# example: what a hard negative looks like compared to a random one
cid, claim, pos = pairs[1]
rng = random.Random(0)
print("claim        :", claim)
print("gold         :", corpus[doc_index[pos]]["title"])
print("hard negative:", corpus[doc_index[hard_pool[cid][0]]]["title"])
print("random neg.  :", corpus[doc_index[sample_negatives("random", cid, rng)[0]]]["title"])

440 training pairs from 399 claims
claim        : 32% of liver transplantation programs required patients to discontinue methadone treatment in 2001.
gold         : Liver transplantation and opioid dependence.
hard negative: Linkage with methadone treatment upon release from incarceration: a promising opportunity.
random neg.  : An IL-13 inhibitor blocks the development of hepatic fibrosis during a T-helper type 2-dominated inflammatory response.


## Model, loss and evaluation

**Loss (InfoNCE)**: for each claim, the similarity to its gold abstract should be higher than to every other abstract in the batch. All abstracts of the batch are candidates: the claim's own negatives, and the positives and negatives of the other claims.

$$\mathcal{L}_i=-\log\frac{e^{\,s(q_i,d_i^+)/\tau}}{\sum_{d\in\text{batch}} e^{\,s(q_i,d)/\tau}}$$

One detail specific to this dataset: many claims come in pairs (a claim and its negation) that share the same gold abstract. If both land in one batch, the other claim's positive would wrongly count as a negative, so any candidate that is gold for the claim is masked out.

In [7]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def sync():
    if device == "cuda":
        torch.cuda.synchronize()

def encode(tok, model, texts, max_len):
    # [CLS] embedding, L2-normalised. Gradients flow through this one (used in training).
    batch = tok(texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(device)
    with torch.autocast(device_type=device, enabled=(device == "cuda")):
        cls = model(**batch).last_hidden_state[:, 0]
    return F.normalize(cls.float(), dim=-1)

@torch.no_grad()
def embed_all(tok, model, texts, max_len, batch_size=64):
    # inference version. Texts are processed from longest to shortest so that batches need little padding.
    model.eval()
    order = np.argsort([-len(t) for t in texts])
    out = np.zeros((len(texts), model.config.hidden_size), dtype=np.float32)
    for i in range(0, len(texts), batch_size):
        idx = order[i:i + batch_size]
        out[idx] = encode(tok, model, [texts[j] for j in idx], max_len).cpu().numpy()
    return out

def score_all_splits(tok, model):
    # re-index the corpus with the current weights and score every split
    doc_emb = embed_all(tok, model, doc_texts, DOC_MAX_LEN)
    out = {}
    for split in ["train", "val", "eval"]:
        q_emb = embed_all(tok, model, [QUERY_PREFIX + c["claim"] for c in claims[split]], QUERY_MAX_LEN)
        out[split] = q_emb @ doc_emb.T
    return out

def ndcg(scores, split):
    return evaluate(make_run(scores[split], split), gold_docs(split))["ndcg10"]

In [8]:
def finetune(mode):
    # mode: "random" or "hard". Returns the score matrices of the best epoch (by val nDCG@10) and the per-epoch history.
    set_seed(SEED)
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    model = AutoModel.from_pretrained(BASE_MODEL).to(device)
    if GRAD_CHECKPOINT:
        model.gradient_checkpointing_enable()

    steps_per_epoch = math.ceil(len(pairs) / BATCH_SIZE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP * EPOCHS * steps_per_epoch), EPOCHS * steps_per_epoch)
    scaler = torch.amp.GradScaler(device, enabled=(device == "cuda"))

    order_rng = random.Random(SEED)      # same batch order in both runs
    neg_rng = random.Random(SEED + 1)
    history, best = [], None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        order = list(range(len(pairs)))
        order_rng.shuffle(order)
        losses, t0 = [], time.perf_counter()

        for start in tqdm(range(0, len(order), BATCH_SIZE), desc=f"{mode} epoch {epoch}", leave=False):
            batch = [pairs[i] for i in order[start:start + BATCH_SIZE]]
            cand_ids = [pos for _, _, pos in batch]                       # candidates: positives first ...
            for cid, _, _ in batch:
                cand_ids += sample_negatives(mode, cid, neg_rng)          # ... then every claim's negatives

            q = encode(tok, model, [QUERY_PREFIX + claim for _, claim, _ in batch], QUERY_MAX_LEN)
            d = encode(tok, model, [doc_texts[doc_index[i]] for i in cand_ids], TRAIN_DOC_LEN)
            sims = q @ d.T / TEMPERATURE                                  # (claims, candidates)

            # a candidate that is gold for the claim but is not its own positive must not act as a negative
            mask = torch.zeros_like(sims, dtype=torch.bool)
            for i, (cid, _, _) in enumerate(batch):
                for j, did in enumerate(cand_ids):
                    if j != i and did in train_gold[cid]:
                        mask[i, j] = True
            sims = sims.masked_fill(mask, -1e4)

            loss = F.cross_entropy(sims, torch.arange(len(batch), device=device))   # the positive of claim i is candidate i
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            losses.append(loss.item())
        sync(); train_time = time.perf_counter() - t0

        scores = score_all_splits(tok, model)
        row = {"run": mode, "epoch": epoch, "train_loss": round(float(np.mean(losses)), 4),
               "train_ndcg10": ndcg(scores, "train"), "val_ndcg10": ndcg(scores, "val"), "train_sec": round(train_time)}
        history.append(row)
        print(row)
        if best is None or row["val_ndcg10"] > best["val_ndcg10"]:       # keep the epoch with the best val nDCG@10
            best = {"epoch": epoch, "val_ndcg10": row["val_ndcg10"], "scores": scores}

    del model, optimizer
    if device == "cuda":
        torch.cuda.empty_cache()
    return best, history

## Starting point (epoch 0)

The base model before any training, as the reference for both runs.

In [9]:
if results_log["task2"]["general_model"] == BASE_MODEL:
    # same model as the general retriever of notebook 2, its scores are already saved
    zero_shot = {s: np.load(WORK / f"scores_general_{s}.npy") for s in ["train", "val", "eval"]}
else:
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    model = AutoModel.from_pretrained(BASE_MODEL).to(device)
    zero_shot = score_all_splits(tok, model)
    del model

base_row = {"run": "zero-shot", "epoch": 0, "train_loss": None,
            "train_ndcg10": ndcg(zero_shot, "train"), "val_ndcg10": ndcg(zero_shot, "val"), "train_sec": 0}
base_row

{'run': 'zero-shot',
 'epoch': 0,
 'train_loss': None,
 'train_ndcg10': 0.9213,
 'val_ndcg10': 0.9328,
 'train_sec': 0}

## Run 1: random negatives

In [10]:
best_random, hist_random = finetune("random")
print("best epoch:", best_random["epoch"], "| val nDCG@10 =", best_random["val_ndcg10"])

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

{'run': 'random', 'epoch': 2, 'train_loss': 0.0135, 'train_ndcg10': 0.9474, 'val_ndcg10': 0.9066, 'train_sec': 68}


random epoch 3:   0%|          | 0/28 [00:00<?, ?it/s]

{'run': 'random', 'epoch': 3, 'train_loss': 0.0097, 'train_ndcg10': 0.9529, 'val_ndcg10': 0.9087, 'train_sec': 67}


random epoch 4:   0%|          | 0/28 [00:00<?, ?it/s]

{'run': 'random', 'epoch': 4, 'train_loss': 0.004, 'train_ndcg10': 0.9559, 'val_ndcg10': 0.9122, 'train_sec': 67}
best epoch: 1 | val nDCG@10 = 0.9153


## Run 2: hard negatives

In [11]:
best_hard, hist_hard = finetune("hard")
print("best epoch:", best_hard["epoch"], "| val nDCG@10 =", best_hard["val_ndcg10"])

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

hard epoch 1:   0%|          | 0/28 [00:00<?, ?it/s]

{'run': 'hard', 'epoch': 1, 'train_loss': 0.2011, 'train_ndcg10': 0.9603, 'val_ndcg10': 0.9401, 'train_sec': 70}


hard epoch 2:   0%|          | 0/28 [00:00<?, ?it/s]

{'run': 'hard', 'epoch': 2, 'train_loss': 0.0554, 'train_ndcg10': 0.9843, 'val_ndcg10': 0.9436, 'train_sec': 69}


hard epoch 3:   0%|          | 0/28 [00:00<?, ?it/s]

{'run': 'hard', 'epoch': 3, 'train_loss': 0.0138, 'train_ndcg10': 0.9882, 'val_ndcg10': 0.946, 'train_sec': 69}


hard epoch 4:   0%|          | 0/28 [00:00<?, ?it/s]

{'run': 'hard', 'epoch': 4, 'train_loss': 0.0067, 'train_ndcg10': 0.9901, 'val_ndcg10': 0.9448, 'train_sec': 69}
best epoch: 3 | val nDCG@10 = 0.946


## Results

Per-epoch curves for both runs. How to read them for report Q3:

- **did hard negatives help?** compare the best val nDCG@10 of the two runs (and both against epoch 0)
- **overfitting?** train loss and train nDCG@10 keep improving every epoch by construction. If val nDCG@10 peaks early and then drops while the train numbers keep getting better, the model is fitting the training claims at the expense of generalisation. The best-val epoch is the one that is kept, which is early stopping.

In [12]:
history = pd.DataFrame([base_row] + hist_random + hist_hard)
history.pivot(index="epoch", columns="run", values=["val_ndcg10", "train_ndcg10", "train_loss"])

val_ndcg10                   train_ndcg10                   train_loss  \
run         hard  random zero-shot         hard  random zero-shot       hard   
epoch                                                                          
0            NaN     NaN    0.9328          NaN     NaN    0.9213        NaN   
1         0.9401  0.9153       NaN       0.9603  0.9433       NaN     0.2011   
2         0.9436  0.9066       NaN       0.9843  0.9474       NaN     0.0554   
3         0.9460  0.9087       NaN       0.9882  0.9529       NaN     0.0138   
4         0.9448  0.9122       NaN       0.9901  0.9559       NaN     0.0067   

                         
run    random zero-shot  
epoch                    
0         NaN       NaN  
1      0.0368       NaN  
2      0.0135       NaN  
3      0.0097       NaN  
4      0.0040       NaN

In [13]:
summary = pd.DataFrame({
    "zero-shot": {"best_epoch": 0, "val_ndcg10": base_row["val_ndcg10"], "train_ndcg10": base_row["train_ndcg10"]},
    "random negatives": {"best_epoch": best_random["epoch"], "val_ndcg10": best_random["val_ndcg10"],
                         "train_ndcg10": ndcg(best_random["scores"], "train")},
    "hard negatives": {"best_epoch": best_hard["epoch"], "val_ndcg10": best_hard["val_ndcg10"],
                       "train_ndcg10": ndcg(best_hard["scores"], "train")},
}).T
print(summary)
print(f"\nhard vs random   : {best_hard['val_ndcg10'] - best_random['val_ndcg10']:+.4f} val nDCG@10")
print(f"random vs zero-shot: {best_random['val_ndcg10'] - base_row['val_ndcg10']:+.4f}")
print(f"hard vs zero-shot  : {best_hard['val_ndcg10'] - base_row['val_ndcg10']:+.4f}")

for name, hist, best in [("random", hist_random, best_random), ("hard", hist_hard, best_hard)]:
    last = hist[-1]
    print(f"{name}: val peaks at epoch {best['epoch']} ({best['val_ndcg10']}), last epoch val = {last['val_ndcg10']}, "
          f"train nDCG@10 at last epoch = {last['train_ndcg10']}")

                  best_epoch  val_ndcg10  train_ndcg10
zero-shot                0.0      0.9328        0.9213
random negatives         1.0      0.9153        0.9433
hard negatives           3.0      0.9460        0.9882

hard vs random   : +0.0307 val nDCG@10
random vs zero-shot: -0.0175
hard vs zero-shot  : +0.0132
random: val peaks at epoch 1 (0.9153), last epoch val = 0.9122, train nDCG@10 at last epoch = 0.9559
hard: val peaks at epoch 3 (0.946), last epoch val = 0.9448, train nDCG@10 at last epoch = 0.9901


In [14]:
# run files (best epoch of each run) + cached scores for task 5
for name, best in [("random", best_random), ("hard", best_hard)]:
    for split in ["train", "val", "eval"]:
        np.save(WORK / f"scores_ft_{name}_{split}.npy", best["scores"][split].astype(np.float32))
    write_trec(PRED / f"t4_{name}.trec", make_run(best["scores"]["eval"], "eval"), f"ft_{name}")

sys.path.append(str(ROOT))
from check_format import check_trec

eval_ids = {c["id"] for c in claims["eval"]}
check_trec(PRED / "t4_random.trec", eval_ids, set(doc_ids.tolist()))
check_trec(PRED / "t4_hard.trec", eval_ids, set(doc_ids.tolist()))
print("t4_random.trec and t4_hard.trec pass check_format")

t4_random.trec and t4_hard.trec pass check_format


In [15]:
results_log["task4"] = {
    "base_model": BASE_MODEL,
    "val_ndcg10_random_negatives": float(best_random["val_ndcg10"]),
    "val_ndcg10_hard_negatives": float(best_hard["val_ndcg10"]),
    "val_ndcg10_zero_shot": float(base_row["val_ndcg10"]),
    "best_epoch": {"random": best_random["epoch"], "hard": best_hard["epoch"]},
    "history": json.loads(history.to_json(orient="records")),
    "settings": {"loss": "InfoNCE, in-batch + explicit negatives, cosine", "epochs": EPOCHS, "batch_size": BATCH_SIZE,
                 "negatives_per_claim": N_NEG, "lr": LR, "temperature": TEMPERATURE, "train_doc_len": TRAIN_DOC_LEN,
                 "hard_negative_pool": f"top {HARD_POOL} non-gold abstracts of RRF(bm25, general, scientific) on the training claims",
                 "seed": SEED, "training_pairs": len(pairs)},
}
results_log["seed"] = SEED
json.dump(results_log, open(WORK / "results_log.json", "w"), indent=2)
{k: results_log["task4"][k] for k in ["base_model", "val_ndcg10_zero_shot", "val_ndcg10_random_negatives", "val_ndcg10_hard_negatives", "best_epoch"]}

{'base_model': 'BAAI/bge-large-en-v1.5',
 'val_ndcg10_zero_shot': 0.9328,
 'val_ndcg10_random_negatives': 0.9153,
 'val_ndcg10_hard_negatives': 0.946,
 'best_epoch': {'random': 1, 'hard': 3}}

## Save for the next notebook

Packs data, predictions and work into `NLP_Inter_IIT.zip` and downloads it (colab only). Upload this zip in the first cell of the next notebook.

In [16]:
if "google.colab" in sys.modules:
    zip_path = Path.cwd() / "NLP_Inter_IIT_5.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for item in ["data", "predictions", "work", "check_format.py", "results_template.json"]:
            p = ROOT / item
            if not p.exists():
                continue
            for f in ([p] if p.is_file() else sorted(p.rglob("*"))):
                if f.is_file():
                    z.write(f, Path("NLP_Inter_IIT") / f.relative_to(ROOT))
    print(zip_path.name, f"{zip_path.stat().st_size / 1e6:.0f} MB")
    from google.colab import files
    files.download(str(zip_path))

NLP_Inter_IIT_5.zip 123 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>